In [ ]:
#!pip install --upgrade pip setuptools wheel
#!pip install -r ./../requirements.txt 

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
Using cached setuptools-82.0.0-py3-none-any.whl (1.0 MB)
Using cached wheel-0.46.3-py3-none-any.whl (30 kB)


ERROR: To modify pip, please run the following command:
C:\Users\pc\anaconda3\python.exe -m pip install --upgrade pip setuptools wheel


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

#Lecture des données
X_train = pd.read_csv("../data/X_train_update.csv")
y_train = pd.read_csv("../data/Y_train_CVw08PX.csv")
X_test = pd.read_csv("../data/X_test_update.csv")

# Affichage des informations sur les datasets
print(f"Info X_train : {X_train.info()}")
print(f"Info Y_train : {y_train.info()}")
print(f"Info X_test : {X_test.info()}")

#Affichage des tailles des datasets
print(f"Taille X_train : {X_train.shape}")
print(f"Taille Y_train : {y_train.shape}")
print(f"Taille X_test : {X_test.shape}")

# Affichage du nombre de classes dans la variable cible
#print(y_train['prdtypecode'].nunique())  # 27 classes
#Merge données d'entrainement
full_data = pd.merge(X_train, y_train, left_index=True, right_index=True)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   84916 non-null  int64 
 1   designation  84916 non-null  object
 2   description  55116 non-null  object
 3   productid    84916 non-null  int64 
 4   imageid      84916 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 3.2+ MB
Info X_train : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Unnamed: 0   84916 non-null  int64
 1   prdtypecode  84916 non-null  int64
dtypes: int64(2)
memory usage: 1.3 MB
Info Y_train : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13812 entries, 0 to 13811
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1

In [3]:
full_data.head()

,Unnamed: 0_x,designation,description,productid,imageid,Unnamed: 0_y,prdtypecode
0,0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,0,10
1,1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,1,2280
2,2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,2,50
3,3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,3,1280
4,4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,4,2705


In [4]:


#Suppression de la colonne Unnamed: 0_y qui est une colonne d'index inutile
full_data = full_data.drop(['Unnamed: 0_y'], axis=1)

#Renomage de la colonne Unnamed: 0_x en id et mise en index de cette colonne
full_data.rename(columns={'Unnamed: 0_x': 'id'}, inplace=True)
full_data.set_index(['id'], inplace=True)

X_test.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
X_test.set_index(['id'], inplace=True)

In [5]:
#Fonction de nettoyage du texte
import re

def clean_text(text):
    if pd.isnull(text):
        return ""

    # Supprimer la ponctuation
    #text = re.sub(r'[^\w\s]', '', text)

    # Supprimer balises HTML
    text = re.sub(r'<.*?>', '', text)

    # Remplacer <br /> par un espace
    text = text.replace(r'<br />', ' ')

    # Remplacer les référence de caractère HTML
    text = text.replace(r'&amp;', '&')
    text = text.replace(r'&nbsp;', ' ')
    text = text.replace(r'&lt', '<')
    text = text.replace(r'&gt', '>')
    text = text.replace(r'&quot', '"')
    text = text.replace(r'&#39', "'")
    text = text.replace(r'&eacute', 'e')
    text = text.replace(r'&egrave', 'e')
    text = text.replace(r'&ecirc', 'e')
    
    # Convertir en minuscules
    text = text.lower()

    # Supprimer les espaces supplémentaires
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [6]:
#Nettoyage du texte
full_data['clean_designation'] = full_data['designation'].apply(clean_text)
full_data['clean_description'] = full_data['description'].apply(clean_text)

#Vérification de doublon dans les désignations et descriptions nettoyées
duplicate_designation = full_data['clean_designation'].duplicated().sum()
duplicate_description = full_data['clean_description'].duplicated().sum()

#Concaténation des désignations et descriptions nettoyées pour l'analyse de texte
full_data['text'] = full_data['clean_designation'] + ' ' + full_data['clean_description']

# Création d'un nouveau dataframe pour l'analyse de texte
test_data = full_data.drop(columns=['designation', 'description', 'productid', 'imageid','clean_designation','clean_description'], axis=1)
display(test_data.head(20))

,prdtypecode,text
id,,
0,10,olivia: personalisiertes notizbuch / 150 seite...
1,2280,journal des arts (le) n° 133 du 28/09/2001 - l...
2,50,grand stylet ergonomique bleu gamepad nintendo...
3,1280,peluche donald - europe - disneyland 2000 (mar...
4,2705,la guerre des tuques luc a des ide;es de grand...
5,2280,afrique contemporaine n° 212 hiver 2004 - doss...
6,10,christof e: bildungsprozessen auf der spur
7,2522,conquérant sept cahier couverture polypro 240 ...
8,1280,puzzle scooby-doo avec poster 2x35 pieces


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

# Importation de TfidfVectorizer pour la vectorisation du texte
vectorizer = TfidfVectorizer()

# Création des ensembles d'entraînement et de test pour l'analyse de texte (20% pour le test)
X_train, X_test, y_train, y_test = train_test_split(full_data['text'], full_data['prdtypecode'], test_size=0.2, random_state=42)

# Vectorisation du texte
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [8]:
import numpy as np
import pandas as pd
from sklearn import svm
from sklearn import model_selection
from sklearn.model_selection import train_test_split
from sklearn import preprocessing



# Standardisation des données de notre dataframe 
Y_train_scaled = preprocessing.scale(X_train)

print(Y_train_scaled.mean(axis=0))
 
print(Y_train_scaled.std(axis=0))
# standardisation du dataframe 
scaler = preprocessing.StandardScaler().fit(Y_train)

Y_train_scaled = scaler.transform(Y_train)


Y_test_scaled = scaler.transform(Y_test)
print(Y_test_scaled.std(axis=0))
 
print(Y_test_scaled.mean(axis=0))



ValueError: Cannot center sparse matrices: pass `with_mean=False` instead See docstring for motivation and alternatives.